In [1]:
# TFT is most useful when you have more data and multiple features.

In [1]:
import pandas as pd

In [2]:
# Important TFT parameters
# The most important parameters in the model above are:

# Parameter	                     Meaning	                                   Starting value
# max_encoder_length	         Historical observations used	               30
# max_prediction_length	         Future observations predicted	               7 
# hidden_size	                 Main neural-network size	                   32
# lstm_layers	                 Number of LSTM layers	                        2
# attention_head_size	         Number of attention heads	                    4
# dropout	                     Regularization	                                0.1
# hidden_continuous_size	     Processing size for continuous variables	     16
# learning_rate	                 Learning rate	                                0.001
# batch_size	                 Samples per training batch	                      64
# max_epochs	                 Maximum training epochs	                      100
# gradient_clip_val	             Gradient clipping	                               0.1
# loss	                         Training objective	QuantileLoss()
# output_size	                  Number of quantile outputs	                    7


In [3]:
import numpy as np

In [4]:
data=pd.read_csv('electricityConsumptionAndProductioction.csv')

In [5]:
data

,DateTime,Consumption,Production,Nuclear,Wind,Hydroelectric,Oil and Gas,Coal,Solar,Biomass
0,2019-01-01 00:00:00,6352,6527,1395,79,1383,1896,1744,0,30
1,2019-01-01 01:00:00,6116,5701,1393,96,1112,1429,1641,0,30
2,2019-01-01 02:00:00,5873,5676,1393,142,1030,1465,1616,0,30
3,2019-01-01 03:00:00,5682,5603,1397,191,972,1455,1558,0,30
4,2019-01-01 04:00:00,5557,5454,1393,159,960,1454,1458,0,30
...,...,...,...,...,...,...,...,...,...,...
62805,2026-03-14 19:00:00,7132,6937,1339,699,2560,1590,598,0,59
62806,2026-03-14 20:00:00,7027,6827,1339,744,2538,1571,579,0,58
62807,2026-03-14 21:00:00,6615,6612,1341,778,2291,1569,578,0,60
62808,2026-03-14 22:00:00,6063,6421,1335,701,2146,1556,586,0,61


In [6]:
data['DateTime']=pd.to_datetime(data['DateTime'])

In [7]:
data.set_index('DateTime',inplace=True)

In [8]:
data.head()

,Consumption,Production,Nuclear,Wind,Hydroelectric,Oil and Gas,Coal,Solar,Biomass
DateTime,,,,,,,,,
2019-01-01 00:00:00,6352,6527,1395,79,1383,1896,1744,0,30
2019-01-01 01:00:00,6116,5701,1393,96,1112,1429,1641,0,30
2019-01-01 02:00:00,5873,5676,1393,142,1030,1465,1616,0,30
2019-01-01 03:00:00,5682,5603,1397,191,972,1455,1558,0,30
2019-01-01 04:00:00,5557,5454,1393,159,960,1454,1458,0,30


In [9]:
data['hour']=data.index.hour
data['weekdays']=data.index.dayofweek
data['quarter']=data.index.quarter
data['month']=data.index.month
data['year']=data.index.month
data["HourOfWeek"] = (data.index.dayofweek * 24 + data.index.hour)

In [10]:
data.head()

,Consumption,Production,Nuclear,Wind,Hydroelectric,Oil and Gas,Coal,Solar,Biomass,hour,weekdays,quarter,month,year,HourOfWeek
DateTime,,,,,,,,,,,,,,,
2019-01-01 00:00:00,6352,6527,1395,79,1383,1896,1744,0,30,0,1,1,1,1,24
2019-01-01 01:00:00,6116,5701,1393,96,1112,1429,1641,0,30,1,1,1,1,1,25
2019-01-01 02:00:00,5873,5676,1393,142,1030,1465,1616,0,30,2,1,1,1,1,26
2019-01-01 03:00:00,5682,5603,1397,191,972,1455,1558,0,30,3,1,1,1,1,27
2019-01-01 04:00:00,5557,5454,1393,159,960,1454,1458,0,30,4,1,1,1,1,28


In [11]:
# pip install torch lightning pytorch-forecasting

In [12]:
#%pip install --upgrade tensorboard

In [13]:
import torch
import lightning
import pytorch_forecasting

In [14]:
from lightning.pytorch import Trainer

In [15]:
from pytorch_forecasting import TemporalFusionTransformer

In [16]:
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import (EarlyStopping,LearningRateMonitor,ModelCheckpoint)

from pytorch_forecasting import (TimeSeriesDataSet,TemporalFusionTransformer)

In [17]:
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

In [18]:
df=pd.read_csv('electricityConsumptionAndProductioction.csv')

In [19]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
print(df.head())


Shape: (62810, 10)

Columns:
['DateTime', 'Consumption', 'Production', 'Nuclear', 'Wind', 'Hydroelectric', 'Oil and Gas', 'Coal', 'Solar', 'Biomass']

First rows:
              DateTime  Consumption  Production  Nuclear  Wind  Hydroelectric  \
0  2019-01-01 00:00:00         6352        6527     1395    79           1383   
1  2019-01-01 01:00:00         6116        5701     1393    96           1112   
2  2019-01-01 02:00:00         5873        5676     1393   142           1030   
3  2019-01-01 03:00:00         5682        5603     1397   191            972   
4  2019-01-01 04:00:00         5557        5454     1393   159            960   

   Oil and Gas  Coal  Solar  Biomass  
0         1896  1744      0       30  
1         1429  1641      0       30  
2         1465  1616      0       30  
3         1455  1558      0       30  
4         1454  1458      0       30  


In [20]:
df["DateTime"] = pd.to_datetime(
    df["DateTime"],
    errors="coerce"
)

In [21]:
df = df.sort_values("DateTime").reset_index(drop=True)

In [22]:
df = df.drop_duplicates(
    subset=["DateTime"]
).reset_index(drop=True)

In [23]:
df.shape

(62799, 10)

In [24]:
import warnings
warnings.filterwarnings("ignore")

In [25]:
numeric_columns = [
    "Consumption",
    "Production",
    "Nuclear",
    "Wind",
    "Hydroelectric",
    "Oil and Gas",
    "Coal",
    "Solar",
    "Biomass"
]

In [26]:
for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).astype("float32")

In [27]:
df[numeric_columns] = (
    df[numeric_columns]
    .interpolate(method="linear")
    .ffill()
    .bfill()
)


In [28]:
df["hour"] = df["DateTime"].dt.hour

In [29]:
df["day"] = df["DateTime"].dt.day

In [30]:
df["day_of_week"] = df["DateTime"].dt.dayofweek

In [31]:
df["day_of_week"] = df["DateTime"].dt.dayofweek
df['month']=df["DateTime"].dt.month

In [32]:
df["year"] = df["DateTime"].dt.year
df["week_of_year"] = (df["DateTime"].dt.isocalendar().week.astype(int))

In [33]:
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [34]:
df["hour_sin"] = np.sin( 2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)


In [35]:
df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

In [36]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

In [37]:
df["time_idx"] = np.arange(len(df))
df["series"] = "Romania"

In [38]:
df["hour"] = df["hour"].astype(str)

df["day_of_week"] = df["day_of_week"].astype(str)

df["month"] = df["month"].astype(str)

df["is_weekend"] = df["is_weekend"].astype(str)

In [39]:
df[['hour','day_of_week','month','is_weekend']].head()

,hour,day_of_week,month,is_weekend
0,0,1,1,0
1,1,1,1,0
2,2,1,1,0
3,3,1,1,0
4,4,1,1,0


In [40]:
ENCODER_LENGTH = 168
PREDICTION_LENGTH = 24
BATCH_SIZE = 64
MAX_EPOCHS = 100

In [41]:
n = len(df)

In [42]:
train_end = int(n * 0.70)
validation_end = int(n * 0.85)

In [43]:
train_df=df.loc[:train_end].copy()
validation_df = df.iloc[train_end - ENCODER_LENGTH: validation_end].copy()
test_df = df.iloc[validation_end - ENCODER_LENGTH:].copy()

In [44]:

print("\nTrain:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


Train: (43960, 25)
Validation: (9588, 25)
Test: (9588, 25)


In [45]:
train_df.head()

,DateTime,Consumption,Production,Nuclear,Wind,Hydroelectric,Oil and Gas,Coal,Solar,Biomass,...,week_of_year,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,time_idx,series
0,2019-01-01 00:00:00,6352.0,6527.0,1395.0,79.0,1383.0,1896.0,1744.0,0.0,30.0,...,1,0,0.000000,1.000000,0.781831,0.62349,0.5,0.866025,0,Romania
1,2019-01-01 01:00:00,6116.0,5701.0,1393.0,96.0,1112.0,1429.0,1641.0,0.0,30.0,...,1,0,0.258819,0.965926,0.781831,0.62349,0.5,0.866025,1,Romania
2,2019-01-01 02:00:00,5873.0,5676.0,1393.0,142.0,1030.0,1465.0,1616.0,0.0,30.0,...,1,0,0.500000,0.866025,0.781831,0.62349,0.5,0.866025,2,Romania
3,2019-01-01 03:00:00,5682.0,5603.0,1397.0,191.0,972.0,1455.0,1558.0,0.0,30.0,...,1,0,0.707107,0.707107,0.781831,0.62349,0.5,0.866025,3,Romania
4,2019-01-01 04:00:00,5557.0,5454.0,1393.0,159.0,960.0,1454.0,1458.0,0.0,30.0,...,1,0,0.866025,0.500000,0.781831,0.62349,0.5,0.866025,4,Romania


In [46]:
training = TimeSeriesDataSet(
    train_df,
    # Time index
    time_idx="time_idx",
    # Target
    target="Consumption",
    # Series identifier
    group_ids=["series"],
    # Historical window
    max_encoder_length=ENCODER_LENGTH,
    # Forecast horizon
    max_prediction_length=PREDICTION_LENGTH,
    # --------------------------------------------------------
    # STATIC CATEGORICAL VARIABLES
    # --------------------------------------------------------

    static_categoricals=[
        "series"
    ],
    # --------------------------------------------------------
    # KNOWN CATEGORICAL VARIABLES
    # --------------------------------------------------------
    time_varying_known_categoricals=[
        "hour",
        "day_of_week",
        "month",
        "is_weekend"
    ],
    # --------------------------------------------------------
    # KNOWN REAL VARIABLES
    # --------------------------------------------------------

    time_varying_known_reals=[
        "time_idx",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos"
    ],

    # --------------------------------------------------------
    # UNKNOWN REAL VARIABLES
    # --------------------------------------------------------

    time_varying_unknown_reals=[
        "Consumption",
        "Production",
        "Nuclear",
        "Wind",
        "Hydroelectric",
        "Oil and Gas",
        "Coal",
        "Solar",
        "Biomass"
    ],


    # --------------------------------------------------------
    # TARGET NORMALIZATION
    # --------------------------------------------------------

    target_normalizer=GroupNormalizer(
        groups=["series"],
        transformation="softplus"
    ),


    # --------------------------------------------------------
    # ALLOW MISSING TIMESTEPS
    # --------------------------------------------------------

    allow_missing_timesteps=True
)


In [47]:
validation = TimeSeriesDataSet.from_dataset(

    training,

    validation_df,

    predict=False,

    stop_randomization=True
)


In [48]:
testing = TimeSeriesDataSet.from_dataset(

    training,

    test_df,

    predict=True,

    stop_randomization=True
)

In [49]:
train_loader = training.to_dataloader(

    train=True,

    batch_size=BATCH_SIZE,

    num_workers=0
)

In [50]:
validation_loader = validation.to_dataloader(

    train=False,

    batch_size=BATCH_SIZE,

    num_workers=0
)

In [51]:
test_loader = testing.to_dataloader(

    train=False,

    batch_size=BATCH_SIZE,

    num_workers=0
)

In [52]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=8,

    min_delta=1e-4,

    mode="min"
)

In [53]:
checkpoint = ModelCheckpoint(

    monitor="val_loss",

    mode="min",

    save_top_k=1,

    filename="best-tft"
)
lr_monitor = LearningRateMonitor(

    logging_interval="epoch"
)

In [54]:

trainer = Trainer(

    max_epochs=MAX_EPOCHS,

    accelerator="auto",

    devices=1,

    gradient_clip_val=0.1,

    callbacks=[
        early_stop,
        checkpoint,
        lr_monitor
    ],

    log_every_n_steps=10
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [55]:
tft = TemporalFusionTransformer.from_dataset(

    training,

    # Neural network size
    hidden_size=32,

    # LSTM layers
    lstm_layers=2,

    # Attention heads
    attention_head_size=4,

    # Dropout
    dropout=0.1,

    # Continuous variable processing
    hidden_continuous_size=16,

    # Quantiles
    output_size=7,

    # Loss
    loss=QuantileLoss(),

    # Learning rate
    learning_rate=0.001,

    # Reduce LR when validation stops improving
    reduce_on_plateau_patience=4,

    # Logging
    log_interval=10
)

In [56]:
print("\nTFT MODEL")
print(tft)

print(
    "\nNumber of parameters:",
    tft.size()
)


TFT MODEL
TemporalFusionTransformer(
  	"attention_head_size":               4
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'Consumption', 'group_ids': ['series'], 'weight': None, 'max_encoder_length': 168, 'min_encoder_length': 168, 'min_prediction_idx': np.int64(0), 'min_prediction_length': 24, 'max_prediction_length': 24, 'static_categoricals': ['series'], 'static_reals': None, 'time_varying_known_categoricals': ['hour', 'day_of_week', 'month', 'is_weekend'], 'time_varying_known_reals': ['time_idx', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['Consumption', 'Production', 'Nuclear', 'Wind', 'Hydroelectric', 'Oil and Gas', 'Coal', 'Solar', 'Biomass'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative

In [57]:
trainer.fit(

    tft,

    train_dataloaders=train_loader,

    val_dataloaders=validation_loader
)

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    326 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    512 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │     96 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 37.1 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 15.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 16.9 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 16.9 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    231 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 122 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 122 K                                                                                                
Total estimated model params size (MB): 0.491                                                                      
Modules in train mode: 526                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [ ]:
import tensorboard
print("numpy:", np.__version__)
print("tensorboard:", tensorboard.__version__)
print("torch:", torch.__version__)
print("lightning:", lightning.__version__)

In [100]:
best_model = TemporalFusionTransformer.load_from_checkpoint(

    checkpoint.best_model_path
)


print(
    "\nBest model:",
    checkpoint.best_model_path
)

PermissionError: [Errno 13] Permission denied: 'D:/ml playlist/time series data'

In [ ]:
predictions = best_model.predict(

    test_loader,

    mode="prediction"
)

In [ ]:
predictions = predictions.detach().cpu().numpy()

In [ ]:
actuals = []

for batch in test_loader:

    x, y = batch

    actual = y[0]

    actuals.append(
        actual
    )


actuals = torch.cat(
    actuals
).detach().cpu().numpy()

In [ ]:
pred = predictions.reshape(-1)

actual = actuals.reshape(-1)

In [ ]:
minimum_length = min(
    len(pred),
    len(actual)
)

pred = pred[:minimum_length]

actual = actual[:minimum_length]

In [ ]:
mae = mean_absolute_error(
    actual,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        pred
    )
)

r2 = r2_score(
    actual,
    pred
)

mape = np.mean(
    np.abs(
        (actual - pred)
        / np.maximum(
            np.abs(actual),
            1e-8
        )
    )
) * 100

In [ ]:
print("\n==============================")
print("TFT RESULTS")
print("==============================")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

In [ ]:
plt.figure(
    figsize=(15, 6)
)

plt.plot(
    actual[:500],
    label="Actual",
    linewidth=2
)

plt.plot(
    pred[:500],
    label="TFT Predicted",
    linewidth=2
)

plt.title(
    "TFT Electricity Consumption Forecast"
)

plt.xlabel("Time")

plt.ylabel("Electricity Consumption")

plt.legend()

plt.grid()

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(7, 7)
)

plt.scatter(
    actual,
    pred,
    alpha=0.3
)

plt.xlabel(
    "Actual Consumption"
)

plt.ylabel(
    "Predicted Consumption"
)

plt.title(
    "Actual vs Predicted"
)

plt.grid()

plt.tight_layout()

plt.show()